In [ ]:
#cleaning phase 
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/stock_data_2025.csv')
df_abc = pd.read_csv('data/raw/stock_data_2024.csv')
df_xyz = pd.read_csv('data/raw/stock_data_2023.csv')
df_22 = pd.read_csv('data/raw/stock_data_2022.csv')
df_21 = pd.read_csv('data/raw/stock_data_2021.csv')
df_20 = pd.read_csv('data/raw/stock_data_2020.csv')

df.replace('-',np.nan,inplace=True)
df_abc.replace('-',np.nan,inplace=True)
df_xyz.replace('-',np.nan,inplace=True)
df_22.replace('-',np.nan,inplace=True)
df_21.replace('-',np.nan,inplace=True)
df_20.replace('-',np.nan,inplace=True)

df.columns = df.columns.str.replace(r'\s+$', '', regex=True)
df_abc.columns = df_abc.columns.str.replace(r'\s+$', '', regex=True)
df_xyz.columns = df_xyz.columns.str.replace(r'\s+$', '', regex=True)
df_22.columns = df_22.columns.str.replace(r'\s+$', '', regex=True)
df_21.columns = df_21.columns.str.replace(r'\s+$', '', regex=True)
df_20.columns = df_20.columns.str.replace(r'\s+$', '', regex=True)

df['Date'] = pd.to_datetime(df['Date'])
df_abc['Date'] = pd.to_datetime(df_abc['Date'])
df_xyz['Date'] = pd.to_datetime(df_xyz['Date'])
df_22['Date'] = pd.to_datetime(df_22['Date'])
df_21['Date'] = pd.to_datetime(df_21['Date'])
df_20['Date'] = pd.to_datetime(df_20['Date'])



pd.set_option('display.max_rows',25)
pd.set_option('display.max_columns',23)


In [ ]:
df.dropna(inplace=True)
df

In [ ]:
df_abc.dropna(inplace=True)
df_abc

In [ ]:
df_xyz.dropna(inplace=True)
df_xyz

In [ ]:
df_22.dropna(inplace=True)
df_22

In [ ]:
df_21.dropna(inplace=True)
df_21

In [ ]:
df_20.dropna(inplace=True)
df_20

In [ ]:
df_all = pd.concat([df, df_abc,df_xyz,df_22,df_21,df_20], ignore_index=True)

In [ ]:
df_all = df_all.sort_values(by='Date')
df_all

In [ ]:
df_all.to_csv('data/processed/final_data_1.csv')

In [ ]:
import pandas as pd

df_me = pd.read_csv('data/processed/final_data_1.csv')

df_me['Date'] = pd.to_datetime(df_me['Date'], format='%Y-%m-%d')

df_me

In [ ]:
df_me=df_me.drop(columns=['Unnamed: 0'])

In [ ]:
df_me

In [ ]:
cols = ['Prev Close','Open Price','High Price','Low Price','Last Price','Close Price','Average Price','Total Traded Quantity','Turnover ₹','No. of Trades','Deliverable Qty']
df_me[cols] = df_me[cols].replace(',', '', regex=True)

In [ ]:
df_me

In [ ]:
df_me[['Prev Close','Open Price','High Price','Low Price','Last Price','Close Price','Average Price','Total Traded Quantity','Turnover ₹','No. of Trades','Deliverable Qty']] = df_me[['Prev Close','Open Price','High Price','Low Price','Last Price','Close Price','Average Price','Total Traded Quantity','Turnover ₹','No. of Trades','Deliverable Qty']].astype(float)

In [ ]:
df_me.info()

In [ ]:
#feature engineering
def calculate_rsi (series,period=14):
    difference = series.diff()

    gain = difference.clip(lower=0)
    loss = -difference.clip(upper=0)

    avg_gain = gain.ewm(alpha = 1/period,adjust=False).mean()
    avg_loss = loss.ewm(alpha =1/period,adjust=False).mean()

    rs = avg_gain/avg_loss

    rsi = 100 - (100/(1+rs))

    return rsi

In [ ]:
df_me['RSI_14'] = calculate_rsi(df_me['Close Price'])

In [ ]:
df_me

In [ ]:
df_me['EMA_12'] = df_me['Close Price'].ewm(span=12,adjust=False).mean()

df_me['EMA_26'] = df_me['Close Price'].ewm(span=26,adjust=False).mean()

df_me['MACD'] = df_me['EMA_12'] - df_me['EMA_26']

df_me['MACD_Signal'] = df_me['MACD'].ewm(span=9,adjust=False).mean()

df_me['MACD_Hist'] = df_me['MACD'] - df_me['MACD_Signal']

df_me["Volume_Change"] = df_me["Total Traded Quantity"].pct_change()

df_me['Target'] = (df_me['Close Price'].shift(-1) > df_me['Close Price']).astype(int)

In [ ]:
import numpy as np

df_me["Volume_Change"] = df_me["Volume_Change"].replace([np.inf, -np.inf], np.nan)

lower = df_me["Volume_Change"].quantile(0.01)
upper = df_me["Volume_Change"].quantile(0.99)

df_me["Volume_Change"] = df_me["Volume_Change"].clip(lower, upper)

In [ ]:
df_me

In [ ]:
df_me['RSI_14'].describe()

In [ ]:
# feature validation
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.plot(df_me["Date"], df_me["Close Price"])
plt.title("Close Price")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(df_me["Date"], df_me["RSI_14"])
plt.axhline(70, linestyle="--")
plt.axhline(30, linestyle="--")
plt.title("RSI (14)")
plt.show()


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df_me["Date"], df_me["Close Price"], label="Close")
plt.plot(df_me["Date"], df_me["EMA_12"], label="EMA 12")
plt.plot(df_me["Date"], df_me["EMA_26"], label="EMA 26")
plt.legend()
plt.title("Close Price vs EMA")
plt.show()


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df_me["Date"], df_me["MACD_Hist"])
plt.axhline(0, linestyle="--")
plt.title("MACD Histogram")
plt.show()


In [ ]:
df_me['Volume_Change'].describe()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df_me["Date"], df_me["Volume_Change"])
plt.title("Volume Change")
plt.show()

In [ ]:
df_me

In [ ]:
df_me.to_csv('data/processed/final_validated_data.csv')

In [ ]:
df_me['Target'].value_counts(normalize=True)

In [ ]:
df_me.dropna(inplace=True)
df_me

In [ ]:
df_final_ml = df_me[["Date","Close Price","RSI_14","EMA_12","EMA_26","MACD_Hist","Volume_Change","Target"]]

In [ ]:
df_final_ml

In [ ]:
df_final_ml.to_csv('data/processed/final_data_ml_ready.csv')

In [ ]:
#news collection 
import pandas as pd 
import requests

url = ('https://newsapi.org/v2/everything?'
       'q=reliance industries&'
       'language=en&'
       'sortBy=popularity&'
       'apiKey=f16b1fd1909946edaf7a8983eb98abed')

response = requests.get(url)
print("Status Code:", response.status_code)


In [ ]:
data = response.json()
if data["status"] != "ok":
  print("Error:", data)
else:
    print("Request successful!")

In [ ]:
articles = data['articles']

df_news = pd.DataFrame(articles)

df_news = df_news[['publishedAt','title','description','source','url']]

df_news["Date"] = pd.to_datetime(df_news["publishedAt"]).dt.date

df_news['Date'] = pd.to_datetime(df_news['Date'])

df_news.columns

In [ ]:
df_news.to_csv("data/raw/reliance_news.csv")

In [ ]:
#news data cleaning 
df_news["text"] = (df_news["title"].fillna("") + " " +df_news["description"].fillna("") )


keywords = ["reliance stock","reliance industries"]
pattern = "|".join(keywords)

df_news = df_news[df_news["text"].str.contains(pattern, case=False, na=False)]

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?\s*:\s*//\S+", "", text)   
    text = re.sub(r"[^a-z\s]", "", text)           
    text = re.sub(r"\s+", " ", text).strip()        
    return text

df_news

In [ ]:
df_news['clean_text'] = df_news['text'].apply(clean_text)
df_news

In [ ]:
df_news = df_news[["Date", "clean_text"]] #the Date column is not in date time format of python please convert and make it the index

In [ ]:
df_news.to_csv("data/processed/reliance_news_cleaned.csv")